# Wild_Chat1M Dataset Preprocessing & EDA

This notebook preprocesses the Wild_Chat1M dataset stored in Google Drive using the `filter_data.py` script.

**Optimizations for Large Files in Colab:**
- Copy file from Drive to Colab's local storage to avoid RAM issues
- Stream processing for large JSONL files
- Memory-efficient batch operations

**Steps:**
1. Mount Google Drive and copy dataset locally
2. Import preprocessing functions from `filter_data.py`
3. Test preprocessing on sample (1000 conversations)
4. Apply preprocessing to full dataset
5. Save to `data/filtered/` (sample & full)
6. Exploratory Data Analysis

## 1. Setup Environment

In [ ]:
# Install required packages
!pip install pandas numpy tqdm -q

import json
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from tqdm.auto import tqdm
import gc  # For garbage collection
import os

print("✓ Packages imported successfully")

## 2. Mount Google Drive & Copy Dataset Locally

⚠️ **CRITICAL:** Copying the file from Drive to local storage prevents RAM overflow issues.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted successfully")

In [ ]:
# ⚠️ CHANGE THIS PATH to match your Google Drive location
drive_path = "/content/drive/MyDrive/wildchat_1M.csv"  # <-- UPDATE THIS PATH!
local_path = "/content/wildchat_1M.csv"  # Local copy in Colab

print(f"Copying file from Drive to local storage...")
print(f"Source: {drive_path}")
print(f"Destination: {local_path}")
print("⏳ This may take a few minutes for large files...")

# Copy file using shell command
!cp "{drive_path}" "{local_path}"

print(f"\n✓ File copied successfully!")

# Check file size
file_size_gb = os.path.getsize(local_path) / (1024**3)
print(f"📊 File size: {file_size_gb:.2f} GB")

## 3. Import Preprocessing Functions from filter_data.py

We'll clone/upload the repository and import the preprocessing functions.

In [ ]:
# Option 1: If you have the repo in Drive, copy it
# !cp -r "/content/drive/MyDrive/llm-empathy-causal-study" "/content/"

# Option 2: Clone from GitHub (if public)
# !git clone https://github.com/dangolofrancesco/llm-empathy-causal-study.git

# Option 3: Upload scripts folder to Colab Files, then:
# Add scripts directory to Python path
sys.path.insert(0, '/content/llm-empathy-causal-study/scripts')

# Import preprocessing functions from filter_data.py
from filter_data import (
    filter_conversation,
    extract_turn_pairs,
    create_user_id,
    extract_time_of_day
)

print("✓ Preprocessing functions imported from filter_data.py")
print("  - filter_conversation()")
print("  - extract_turn_pairs()")
print("  - create_user_id()")
print("  - extract_time_of_day()")

## 4. Test Preprocessing on Sample (1000 conversations)

Let's test on a small sample first to verify everything works correctly.

In [ ]:
# Process sample (first 1000 conversations)
print("🔍 Processing SAMPLE (1000 conversations)...\n")

sample_turn_pairs = []
sample_size = 1000
conversations_processed = 0
conversations_filtered = 0

# Read JSONL file line by line (memory efficient)
with open(local_path, 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(tqdm(f, total=sample_size, desc="Sample processing"), 1):
        if line_num > sample_size:
            break
        
        try:
            record = json.loads(line)
            
            conversation_hash = record.get('conversation_hash', '')
            model = record.get('model', '')
            timestamp = record.get('timestamp', '')
            conversation = record.get('conversation', [])
            
            # Apply filters using filter_data.py function
            if not filter_conversation(conversation):
                conversations_filtered += 1
                continue
            
            # Extract turn-pairs using filter_data.py function
            turn_pairs = extract_turn_pairs(conversation, conversation_hash, model, timestamp)
            sample_turn_pairs.extend(turn_pairs)
            conversations_processed += 1
            
        except Exception as e:
            if line_num % 100 == 0:
                print(f"⚠️  Warning at line {line_num}: {e}")
            continue

print(f"\n{'='*80}")
print("📊 SAMPLE PREPROCESSING RESULTS")
print(f"{'='*80}")
print(f"Conversations read:      {sample_size:,}")
print(f"Conversations kept:      {conversations_processed:,}")
print(f"Conversations filtered:  {conversations_filtered:,}")
print(f"Turn-pairs extracted:    {len(sample_turn_pairs):,}")
print(f"Filter rate:             {(conversations_filtered/sample_size)*100:.1f}%")
print(f"{'='*80}")

In [ ]:
# Create DataFrame and examine sample
df_sample = pd.DataFrame(sample_turn_pairs)

print(f"📋 Sample DataFrame shape: {df_sample.shape}")
print(f"\n📊 Columns ({len(df_sample.columns)}):")
for col in df_sample.columns:
    print(f"  • {col}")

print(f"\n🔍 First 3 rows:")
display(df_sample.head(3))

print(f"\n{'='*80}")
print("📈 SAMPLE STATISTICS")
print(f"{'='*80}")
print(f"Unique conversations:  {df_sample['conversation_hash'].nunique():,}")
print(f"Unique users:          {df_sample['user_id'].nunique():,}")
print(f"Unique models:         {df_sample['model'].nunique()}")

print(f"\n🤖 Model distribution:")
print(df_sample['model'].value_counts())

print(f"\n🕐 Time of day distribution:")
print(df_sample['time_of_day'].value_counts())

print(f"\n📝 Text content lengths (avg characters):")
print(f"  User prompt:        {df_sample['user_prompt_content'].str.len().mean():.0f}")
print(f"  Assistant response: {df_sample['assistant_response_content'].str.len().mean():.0f}")
print(f"  User reply:         {df_sample['user_reply_content'].str.len().mean():.0f}")

print(f"\n💬 Conversation stats:")
print(f"  Avg length:         {df_sample['conversation_length'].mean():.2f} turns")
print(f"  Avg turn position:  {df_sample['turn_position'].mean():.2f}")
print(f"{'='*80}")

## 5. Save Sample Data

In [ ]:
# Create output directory
output_dir = Path('/content/data/filtered')
output_dir.mkdir(parents=True, exist_ok=True)

# Save sample to CSV
sample_output_path = output_dir / 'filtered_turn_pairs_sample.csv'
df_sample.to_csv(sample_output_path, index=False)

print(f"✅ Sample data saved!")
print(f"📁 Path: {sample_output_path}")
print(f"📊 Shape: {df_sample.shape}")
print(f"💾 Size: {sample_output_path.stat().st_size / 1024 / 1024:.2f} MB")

# Also copy to Drive for persistence
drive_output_dir = Path('/content/drive/MyDrive/llm-empathy-causal-study/data/filtered')
drive_output_dir.mkdir(parents=True, exist_ok=True)
df_sample.to_csv(drive_output_dir / 'filtered_turn_pairs_sample.csv', index=False)
print(f"📤 Also saved to Google Drive: {drive_output_dir / 'filtered_turn_pairs_sample.csv'}")

## 6. Full Dataset Preprocessing

Now let's process the entire dataset. ⚠️ **This may take 30+ minutes depending on dataset size.**

In [ ]:
# Clear memory before processing full dataset
del df_sample, sample_turn_pairs
gc.collect()

print("🚀 Processing FULL dataset...")
print("⏰ This may take 30+ minutes for large datasets\n")

all_turn_pairs = []
conversations_processed = 0
conversations_filtered = 0
total_lines = 0

# Read JSONL file line by line (memory efficient)
with open(local_path, 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(tqdm(f, desc="Full processing"), 1):
        total_lines = line_num
        
        try:
            record = json.loads(line)
            
            conversation_hash = record.get('conversation_hash', '')
            model = record.get('model', '')
            timestamp = record.get('timestamp', '')
            conversation = record.get('conversation', [])
            
            # Apply filters using filter_data.py function
            if not filter_conversation(conversation):
                conversations_filtered += 1
                continue
            
            # Extract turn-pairs using filter_data.py function
            turn_pairs = extract_turn_pairs(conversation, conversation_hash, model, timestamp)
            all_turn_pairs.extend(turn_pairs)
            conversations_processed += 1
            
            # Progress update every 10k conversations
            if line_num % 10000 == 0:
                print(f"  📍 {line_num:,} conversations | "
                      f"Kept: {conversations_processed:,} | "
                      f"Turn-pairs: {len(all_turn_pairs):,}")
                
                # Periodic garbage collection to manage memory
                if line_num % 50000 == 0:
                    gc.collect()
            
        except Exception as e:
            if line_num % 10000 == 0:
                print(f"  ⚠️  Warning at line {line_num}: {e}")
            continue

print(f"\n{'='*80}")
print("📊 FULL PREPROCESSING RESULTS")
print(f"{'='*80}")
print(f"Total conversations read:  {total_lines:,}")
print(f"Conversations kept:        {conversations_processed:,}")
print(f"Conversations filtered:    {conversations_filtered:,}")
print(f"Turn-pairs extracted:      {len(all_turn_pairs):,}")
print(f"Filter rate:               {(conversations_filtered/total_lines)*100:.1f}%")
print(f"{'='*80}")

In [ ]:
# Create DataFrame from full dataset
print("📊 Creating DataFrame...")
df_full = pd.DataFrame(all_turn_pairs)

print(f"\n📋 Full DataFrame shape: {df_full.shape}")
print(f"\n🔍 First 3 rows:")
display(df_full.head(3))

print(f"\n{'='*80}")
print("📈 FULL DATASET STATISTICS")
print(f"{'='*80}")
print(f"Unique conversations:  {df_full['conversation_hash'].nunique():,}")
print(f"Unique users:          {df_full['user_id'].nunique():,}")
print(f"Unique models:         {df_full['model'].nunique()}")

print(f"\n🤖 Model distribution:")
print(df_full['model'].value_counts())

print(f"\n🕐 Time of day distribution:")
print(df_full['time_of_day'].value_counts())

print(f"\n📝 Text content lengths (avg characters):")
print(f"  User prompt:        {df_full['user_prompt_content'].str.len().mean():.0f}")
print(f"  Assistant response: {df_full['assistant_response_content'].str.len().mean():.0f}")
print(f"  User reply:         {df_full['user_reply_content'].str.len().mean():.0f}")

print(f"\n💬 Conversation length stats:")
print(df_full['conversation_length'].describe())

print(f"\n📍 Turn position stats:")
print(df_full['turn_position'].describe())
print(f"{'='*80}")

## 7. Save Full Dataset

In [ ]:
# Save full dataset to CSV
full_output_path = output_dir / 'filtered_turn_pairs_full.csv'
print(f"💾 Saving full dataset to: {full_output_path}")

df_full.to_csv(full_output_path, index=False)

print(f"\n✅ Full data saved!")
print(f"📁 Path: {full_output_path}")
print(f"📊 Shape: {df_full.shape}")
print(f"💾 Size: {full_output_path.stat().st_size / 1024 / 1024:.2f} MB")

# Also save to Google Drive for persistence
drive_full_path = drive_output_dir / 'filtered_turn_pairs_full.csv'
print(f"\n📤 Saving to Google Drive...")
df_full.to_csv(drive_full_path, index=False)
print(f"✅ Saved to Drive: {drive_full_path}")

print(f"\n{'='*80}")
print("🎉 PREPROCESSING COMPLETE!")
print(f"{'='*80}")
print(f"\n📂 Output files:")
print(f"  Sample: {output_dir / 'filtered_turn_pairs_sample.csv'}")
print(f"  Full:   {full_output_path}")
print(f"\n📂 Google Drive backup:")
print(f"  Sample: {drive_output_dir / 'filtered_turn_pairs_sample.csv'}")
print(f"  Full:   {drive_full_path}")
print(f"\n🎯 Next steps:")
print(f"  1. Score conversations with score_conversations.py")
print(f"  2. Create matched pairs with create_matched_pairs.py")
print(f"  3. Analyze causal effects in 03_analysis.ipynb")
print(f"{'='*80}")

## 8. Data Quality Checks

In [ ]:
# Check for missing values
print("🔍 Missing values in full dataset:")
missing = df_full.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print(missing)
else:
    print("✅ No missing values!")

# Check for duplicates
duplicates = df_full['turn_triplet_id'].duplicated().sum()
print(f"\n🔍 Duplicate turn triplet IDs: {duplicates}")
if duplicates == 0:
    print("✅ No duplicates!")

# Check language (should all be English after filtering)
print(f"\n🔍 Language distribution (should all be English):")
lang_counts = df_full['language'].value_counts()
print(lang_counts)
if len(lang_counts) == 1 and lang_counts.index[0].lower() == 'english':
    print("✅ All English!")

# Check for empty content
print(f"\n🔍 Checking for empty content:")
empty_prompts = (df_full['user_prompt_content'].str.len() == 0).sum()
empty_responses = (df_full['assistant_response_content'].str.len() == 0).sum()
empty_replies = (df_full['user_reply_content'].str.len() == 0).sum()

print(f"  Empty user prompts: {empty_prompts}")
print(f"  Empty assistant responses: {empty_responses}")
print(f"  Empty user replies: {empty_replies}")

if empty_prompts + empty_responses + empty_replies == 0:
    print("✅ No empty content!")

print(f"\n{'='*80}")
print("✅ Data quality checks complete!")
print(f"{'='*80}")

## 9. Exploratory Data Analysis (EDA)

Now let's explore the preprocessed data to understand its characteristics.

In [ ]:
# Install visualization libraries if needed
!pip install matplotlib seaborn plotly -q

import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Visualization libraries loaded")

In [ ]:
# Distribution of conversation lengths
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Conversation length
axes[0].hist(df_full['conversation_length'], bins=50, edgecolor='black')
axes[0].set_xlabel('Conversation Length (turns)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Conversation Lengths')
axes[0].axvline(df_full['conversation_length'].median(), color='red', 
                linestyle='--', label=f'Median: {df_full["conversation_length"].median():.0f}')
axes[0].legend()

# Turn position
axes[1].hist(df_full['turn_position'], bins=50, edgecolor='black')
axes[1].set_xlabel('Turn Position')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Turn Positions')
axes[1].axvline(df_full['turn_position'].median(), color='red', 
                linestyle='--', label=f'Median: {df_full["turn_position"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"📊 Conversation length - Mean: {df_full['conversation_length'].mean():.2f}, "
      f"Median: {df_full['conversation_length'].median():.0f}, "
      f"Max: {df_full['conversation_length'].max()}")
print(f"📊 Turn position - Mean: {df_full['turn_position'].mean():.2f}, "
      f"Median: {df_full['turn_position'].median():.0f}, "
      f"Max: {df_full['turn_position'].max()}")

In [ ]:
# Text length distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# User prompt length
axes[0].hist(df_full['user_prompt_content'].str.len(), bins=50, edgecolor='black')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('User Prompt Length Distribution')
axes[0].set_xlim(0, df_full['user_prompt_content'].str.len().quantile(0.95))

# Assistant response length
axes[1].hist(df_full['assistant_response_content'].str.len(), bins=50, edgecolor='black', color='orange')
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Assistant Response Length Distribution')
axes[1].set_xlim(0, df_full['assistant_response_content'].str.len().quantile(0.95))

# User reply length
axes[2].hist(df_full['user_reply_content'].str.len(), bins=50, edgecolor='black', color='green')
axes[2].set_xlabel('Character Count')
axes[2].set_ylabel('Frequency')
axes[2].set_title('User Reply Length Distribution')
axes[2].set_xlim(0, df_full['user_reply_content'].str.len().quantile(0.95))

plt.tight_layout()
plt.show()

print(f"📝 Text length statistics (characters):")
print(f"  User prompt:        {df_full['user_prompt_content'].str.len().describe()}")
print(f"  Assistant response: {df_full['assistant_response_content'].str.len().describe()}")
print(f"  User reply:         {df_full['user_reply_content'].str.len().describe()}")

In [ ]:
# Categorical variable distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Model distribution
model_counts = df_full['model'].value_counts()
axes[0, 0].bar(range(len(model_counts)), model_counts.values)
axes[0, 0].set_xticks(range(len(model_counts)))
axes[0, 0].set_xticklabels(model_counts.index, rotation=45, ha='right')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Model Distribution')
axes[0, 0].grid(axis='y', alpha=0.3)

# Time of day distribution
time_counts = df_full['time_of_day'].value_counts()
axes[0, 1].bar(time_counts.index, time_counts.values, color='skyblue')
axes[0, 1].set_xlabel('Time of Day')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Time of Day Distribution')
axes[0, 1].grid(axis='y', alpha=0.3)

# Top 10 countries
country_counts = df_full['user_country'].value_counts().head(10)
axes[1, 0].barh(range(len(country_counts)), country_counts.values)
axes[1, 0].set_yticks(range(len(country_counts)))
axes[1, 0].set_yticklabels(country_counts.index)
axes[1, 0].set_xlabel('Count')
axes[1, 0].set_title('Top 10 Countries')
axes[1, 0].invert_yaxis()
axes[1, 0].grid(axis='x', alpha=0.3)

# Redacted content distribution
redacted_data = {
    'User Prompt': df_full['user_prompt_redacted'].sum(),
    'Assistant Response': df_full['assistant_response_redacted'].sum(),
    'User Reply': df_full['user_reply_redacted'].sum()
}
axes[1, 1].bar(redacted_data.keys(), redacted_data.values(), color=['blue', 'orange', 'green'])
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Redacted Content Count')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Summary

### Preprocessing Complete! ✅

**Files Created:**
- `data/filtered/filtered_turn_pairs_sample.csv` - Sample dataset for testing
- `data/filtered/filtered_turn_pairs_full.csv` - Full preprocessed dataset

**Key Variables Extracted:**
- **T (Treatment)**: `assistant_response_content` → to be scored for empathy
- **Y (Outcome)**: `user_reply_content` → to be scored for attachment  
- **X1 (Confounder)**: `user_prompt_content` → for matching
- **X2 (Confounder)**: `user_id` → user latent traits
- **X3 (Confounder)**: `turn_position`, `conversation_length`, `time_of_day` → context
- **X4 (Confounder)**: `model` → model ID

**Filters Applied:**
- ✅ Conversations with ≥3 turns
- ✅ English language only
- ✅ Non-toxic content

**Next Steps:**
1. **Score conversations** using `score_conversations.py` to generate empathy and attachment scores
2. **Create matched pairs** using `create_matched_pairs.py` for causal analysis
3. **Analyze causal effects** in `03_analysis.ipynb`